In [15]:
from dotenv import load_dotenv
load_dotenv()
import os
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec


In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pc.create_index(
    name="rag", dimension=1536, metric ="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    # number of dimensions OpenAI embeddings have, which is 1536 for text-embedding-ada-002
    # you can use open source embeddings like sentence-transformers/all-MiniLM-L6-v2 which has 384 dimensions, but try to match
    # uses cosine similarity for vector search, which is a good default for text embeddings
    # spec is the serverless spec for the index, which is a good default for small to medium sized indexes
    # hosted on AWS in the us-east-1 region
)

IndexModel(name='rag', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://rag-3yxgjhj.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [9]:
import json
data = json.load(open('reviews.json'))
data['reviews']

[{'professor': 'Dr. Alan Turing',
  'subject': 'Computer Science',
  'stars': 4,
  'review': 'Great professor overall. Lectures are solid and well-paced, though the exams can be a bit challenging. Just make sure to study.'},
 {'professor': 'Dr. Ada Lovelace',
  'subject': 'Computer Science',
  'stars': 4,
  'review': "Very knowledgeable and approachable during office hours. Homework load is reasonable but don't fall behind."},
 {'professor': 'Dr. Grace Hopper',
  'subject': 'Data Structures',
  'stars': 2,
  'review': 'Hard to follow during lectures. They tend to ramble and go off-topic frequently. Exams are much harder than the practice problems.'},
 {'professor': 'Prof. Richard Feynman',
  'subject': 'Physics',
  'stars': 4,
  'review': 'Great professor overall. Lectures are solid and well-paced, though the exams can be a bit challenging. Just make sure to study.'},
 {'professor': 'Dr. Marie Curie',
  'subject': 'Chemistry',
  'stars': 4,
  'review': 'Great professor overall. Lecture

In [17]:
processed_data = []
client = OpenAI()
# takes the reviews and converts to semantic embeddings using OpenAI's text-embedding-ada-002 model and then upserts to Pinecone
# man and uncle/woman and aunt will be more semantically related than woman and uncle

for review in data['reviews']:
    response = client.embeddings.create(
        input = review['review'],
        model = "text-embedding-3-small",
    )
    embedding = response.data[0].embedding
    processed_data.append({
        "values": embedding,
        "id": review['professor'],
        "metadata": {
            "review": review["review"],
            "subject": review["subject"],
            "stars": review["stars"],   
     }})
        # this is what pinecone suggests. if you want to use another embedding model, make sure your data is in the expected format


In [18]:
processed_data[0]

{'values': [-0.0039520263671875,
  -0.045135498046875,
  -0.032806396484375,
  6.496906280517578e-05,
  -0.0090179443359375,
  0.009796142578125,
  -0.016357421875,
  -0.01137542724609375,
  0.01464080810546875,
  -0.022674560546875,
  0.0008893013000488281,
  0.01605224609375,
  -0.014862060546875,
  -0.0070953369140625,
  -0.00933074951171875,
  0.0211334228515625,
  -0.03472900390625,
  0.004150390625,
  -0.01439666748046875,
  0.005786895751953125,
  0.037078857421875,
  -0.03521728515625,
  0.038299560546875,
  0.015655517578125,
  -0.042022705078125,
  -0.01409149169921875,
  -0.0014867782592773438,
  0.043060302734375,
  0.0179595947265625,
  0.01415252685546875,
  0.067138671875,
  -0.0030841827392578125,
  0.0005078315734863281,
  -0.0401611328125,
  -0.02618408203125,
  0.041717529296875,
  -0.009063720703125,
  0.0292816162109375,
  0.012237548828125,
  0.00988006591796875,
  0.0279388427734375,
  0.02880859375,
  0.0019235610961914062,
  0.0197601318359375,
  0.041046142578

In [19]:
index = pc.Index('rag')
# rag is the index we created
# we can now upsert the data to the index. this will take a few seconds, depending on the size of the data
# upsert vs insert: upsert will update the existing data if the id already exists, insert will throw an error if the id already exists
index.upsert(
    vectors=processed_data,
    namespace="ns1"
    # ns1 = namespace1
    # in firebase, the index is a collection, the namespace is a document, and the vector is a field in the document. this is a good way to organize your data if you have multiple collections of data that you want to keep separate
)

UpsertResponse(upserted_count=20)

In [20]:
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=20, metric='cosine', namespaces=1)